# Phase 3a -- FinBERT News Sentiment for Wheat (GDELT 2.0 + FinBERT)

**Goal:** build a daily news-sentiment feature series for 2017-01-01 -> today, using GDELT 2.0 as the headline corpus and FinBERT (`ProsusAI/finbert`) for sentiment scoring. Output is a single CSV that the v2.4.1 ablation notebook will consume as a new static-feature group.

**Window:** GDELT 2.0 has dense English coverage from ~2017+. Earlier dates have sparse ag-themed records, so we restrict to 2017+ and treat this as the **short-window ablation track** (parallel to the Sentinel-2 / Polymarket constraints).

**Features produced (3 columns, daily):**
- `news_sentiment_mean` -- mean of `(pos_prob - neg_prob)` across the day's wheat-related headlines.
- `news_count` -- number of unique headlines per day (proxy for news volume / attention).
- `news_negative_tail_mean` -- mean sentiment of the most-negative quartile of the day's headlines (captures crisis spikes that the simple mean would dilute).

**Reproducibility:**
- GDELT API queries are deterministic for fixed `(query, time-window)` pairs.
- FinBERT inference is deterministic on GPU with fixed seed + cudnn.deterministic.
- All outputs cached to `Drive/Quants/alternative_data/raw/` and `processed/` -- re-running the ablation later does NOT re-hit GDELT or re-run FinBERT.

**Pipeline:**
1. Mount Drive (same path as v2.4.1).
2. Query GDELT 2.0 DocAPI for wheat-relevant headlines, week-by-week, 2017-01-01 -> today.
3. Deduplicate by URL hash; cache raw headlines CSV.
4. Run FinBERT batched inference on each headline (GPU).
5. Aggregate to daily features.
6. Save to `Drive/Quants/alternative_data/processed/finbert_wheat_sentiment_daily.csv`.

**GPU:** Colab T4 is enough. A100 if you want it faster.
**No data upload required** -- everything is pulled live from GDELT.


## 1. Install deps

In [ ]:
!pip install -q transformers==4.45.2 sentencepiece tqdm


## 2. Imports + seeds

In [ ]:
import os, time, json, urllib.parse
from datetime import datetime, timedelta
from pathlib import Path
import requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print('Device:', DEVICE)


## 3. Drive mount + paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/Quants ')
ALT_RAW  = BASE / 'alternative_data' / 'raw'
ALT_PROC = BASE / 'alternative_data' / 'processed'
ALT_RAW.mkdir(parents=True, exist_ok=True)
ALT_PROC.mkdir(parents=True, exist_ok=True)

RAW_HEADLINES_CSV  = ALT_RAW  / 'gdelt_wheat_headlines_2017_present.csv'
SCORED_CSV         = ALT_RAW  / 'gdelt_wheat_headlines_finbert_scored.csv'
DAILY_FEATURES_CSV = ALT_PROC / 'finbert_wheat_sentiment_daily.csv'
print('Outputs ->', ALT_PROC)


## 4. Pull GDELT 2.0 headlines (week-by-week)

GDELT 2.0 DocAPI returns up to 250 articles per query. We paginate by 7-day windows. Cached to Drive -- skipped on re-run.


In [ ]:
GDELT_URL = 'https://api.gdeltproject.org/api/v2/doc/doc'

QUERY = (
    '("wheat futures" OR "wheat prices" OR "wheat harvest" OR "wheat crop" '
    'OR "wheat exports" OR "wheat shortage" OR "grain market" OR "grain exports" '
    'OR "wheat production" OR "wheat yield" OR "wheat planting") '
    'sourcelang:eng'
)

START_DATE  = datetime(2017, 1, 1)
END_DATE    = datetime.utcnow()
WINDOW_DAYS = 7

def gdelt_pull_window(start, end, max_records=250, retries=3):
    params = {
        'query': QUERY,
        'mode': 'ArtList',
        'format': 'json',
        'maxrecords': max_records,
        'startdatetime': start.strftime('%Y%m%d%H%M%S'),
        'enddatetime':   end.strftime('%Y%m%d%H%M%S'),
        'sort': 'HybridRel',
    }
    url = GDELT_URL + '?' + urllib.parse.urlencode(params)
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                try:
                    return r.json().get('articles', [])
                except json.JSONDecodeError:
                    return []
            else:
                time.sleep(2 ** attempt)
        except requests.RequestException:
            time.sleep(2 ** attempt)
    return []

if RAW_HEADLINES_CSV.exists():
    print(f'Cached headlines found at {RAW_HEADLINES_CSV} -- loading.')
    raw_df = pd.read_csv(RAW_HEADLINES_CSV)
    print(f'  {len(raw_df)} headlines')
else:
    print('No cache -- pulling from GDELT (~5-10 min)...')
    rows = []
    cur = START_DATE
    pbar = tqdm(total=(END_DATE - START_DATE).days // WINDOW_DAYS + 1)
    while cur < END_DATE:
        nxt = min(cur + timedelta(days=WINDOW_DAYS), END_DATE)
        articles = gdelt_pull_window(cur, nxt)
        for a in articles:
            rows.append({
                'seendate': a.get('seendate', ''),
                'title':    a.get('title', '').strip(),
                'url':      a.get('url', ''),
                'domain':   a.get('domain', ''),
                'language': a.get('language', ''),
            })
        cur = nxt
        pbar.update(1)
        time.sleep(0.3)
    pbar.close()
    raw_df = pd.DataFrame(rows)
    raw_df.to_csv(RAW_HEADLINES_CSV, index=False)
    print(f'Pulled {len(raw_df)} raw rows -> {RAW_HEADLINES_CSV}')


## 5. Deduplicate + clean

In [ ]:
raw_df = raw_df.dropna(subset=['title', 'url'])
raw_df = raw_df[raw_df['title'].str.len() > 10]
raw_df['date'] = pd.to_datetime(raw_df['seendate'], format='%Y%m%d%H%M%S', errors='coerce')
raw_df = raw_df.dropna(subset=['date'])

raw_df = raw_df.drop_duplicates(subset=['url'])
raw_df['title_norm'] = raw_df['title'].str.lower().str.strip()
raw_df = raw_df.sort_values('date')
raw_df['day'] = raw_df['date'].dt.date
raw_df = raw_df.drop_duplicates(subset=['title_norm', 'day'])

EXCLUDE_PATTERNS = ['wheat allergy', 'wheat free recipe', 'gluten free',
                    'wheat thins', 'mr wheat', 'mrs wheat']
mask = ~raw_df['title_norm'].apply(lambda s: any(p in s for p in EXCLUDE_PATTERNS))
raw_df = raw_df[mask].reset_index(drop=True)

print(f'After dedup + filter: {len(raw_df)} headlines')
print(f'Date range: {raw_df["date"].min()} .. {raw_df["date"].max()}')
print(f'\nTop domains:\n{raw_df["domain"].value_counts().head(15)}')
print(f'\nDaily headline count summary:\n{raw_df.groupby("day").size().describe()}')


## 6. FinBERT inference (batched on GPU)

`ProsusAI/finbert` outputs 3 logits -> softmax -> `[positive, negative, neutral]` probabilities. Net sentiment = `pos - neg`. Headlines truncated to 64 tokens.


In [ ]:
if SCORED_CSV.exists():
    print(f'Cached scored headlines found -- loading.')
    scored = pd.read_csv(SCORED_CSV, parse_dates=['date'])
else:
    print('Loading FinBERT model...')
    tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
    model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert').to(DEVICE)
    model.eval()
    LABEL_POS, LABEL_NEG, LABEL_NEU = 0, 1, 2

    titles = raw_df['title'].astype(str).tolist()
    BATCH = 64
    pos_p, neg_p, neu_p = [], [], []
    for i in tqdm(range(0, len(titles), BATCH), desc='FinBERT'):
        batch = titles[i:i+BATCH]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=64,
                        return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        pos_p.extend(probs[:, LABEL_POS].tolist())
        neg_p.extend(probs[:, LABEL_NEG].tolist())
        neu_p.extend(probs[:, LABEL_NEU].tolist())

    scored = raw_df.copy()
    scored['pos_prob'] = pos_p
    scored['neg_prob'] = neg_p
    scored['neu_prob'] = neu_p
    scored['sentiment'] = scored['pos_prob'] - scored['neg_prob']
    scored.to_csv(SCORED_CSV, index=False)
    print(f'Saved scored headlines -> {SCORED_CSV}')

print(f'Sentiment summary:\n{scored["sentiment"].describe()}')


## 7. Daily aggregation -> 3 features

In [ ]:
scored['day'] = pd.to_datetime(scored['date']).dt.normalize()

def neg_tail_mean(s):
    if len(s) == 0: return 0.0
    cutoff = np.quantile(s, 0.25)
    tail = s[s <= cutoff]
    return float(tail.mean()) if len(tail) > 0 else float(s.mean())

agg = scored.groupby('day').agg(
    news_sentiment_mean=('sentiment', 'mean'),
    news_count=('sentiment', 'size'),
    news_negative_tail_mean=('sentiment', neg_tail_mean),
).reset_index().rename(columns={'day': 'date'})

# Reindex to a continuous daily calendar; ffill across no-news days; lag 1 day.
full_idx = pd.date_range(agg['date'].min(), agg['date'].max(), freq='D')
agg = agg.set_index('date').reindex(full_idx)
agg['news_count'] = agg['news_count'].fillna(0).astype(int)
agg[['news_sentiment_mean', 'news_negative_tail_mean']] = \
    agg[['news_sentiment_mean', 'news_negative_tail_mean']].ffill().fillna(0.0)

# 1-day publication lag (predict t using info known at t-1)
agg = agg.shift(1).dropna()
agg.index.name = 'date'
agg.to_csv(DAILY_FEATURES_CSV)
print(f'Saved daily features -> {DAILY_FEATURES_CSV}')
print(agg.tail())
print(f'\nShape: {agg.shape}')
print(f'Date range: {agg.index.min().date()} .. {agg.index.max().date()}')


## 8. Sanity plot

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
agg['news_sentiment_mean'].rolling(30).mean().plot(ax=axes[0], color='steelblue')
axes[0].set_title('News sentiment (30-day rolling mean of pos - neg)')
axes[0].axhline(0, color='gray', ls='--', alpha=0.5)
agg['news_count'].rolling(30).mean().plot(ax=axes[1], color='darkorange')
axes[1].set_title('Daily headline count (30-day rolling mean)')
agg['news_negative_tail_mean'].rolling(30).mean().plot(ax=axes[2], color='firebrick')
axes[2].set_title('Negative-tail mean sentiment (30-day rolling mean)')
axes[2].set_xlabel('date')
plt.tight_layout(); plt.show()

print('\nSpot checks (should show sentiment dips / volume spikes):')
events = ['2022-02-24',  # Russia invades Ukraine
          '2022-07-22',  # Black Sea grain deal
          '2023-07-17',  # Russia exits grain deal
          '2024-08-06']  # Ukrainian incursion into Kursk
for e in events:
    ts = pd.Timestamp(e)
    if ts in agg.index:
        row = agg.loc[ts]
        print(f'  {e}: sentiment={row["news_sentiment_mean"]:+.3f}  '
              f'count={int(row["news_count"]):3d}  neg_tail={row["news_negative_tail_mean"]:+.3f}')


## Done

CSV ready at `Drive/Quants/alternative_data/processed/finbert_wheat_sentiment_daily.csv`.

The next notebook (`phase3_ablation_v241.ipynb`, built after this one runs) will:
1. Reuse the v2.4.1 baseline pipeline verbatim.
2. Read this CSV; align to the daily index used by the classifier.
3. Concatenate the 3 sentiment columns onto the static-feature branch.
4. Run the matched 2017+ baseline + the +FinBERT cell across the 3 deep models.

To re-pull from scratch, delete the cached CSVs in `Drive/Quants/alternative_data/raw/`.
